# The hypergraph Turán number of the tetrahedron

In [ ]:
#@title Verification code

"""Initial program for the Turan Tetrahedron problem."""

# pylint: disable=unused-import
# pylint: disable=g-bad-import-order


linprog = optimize.linprog







def get_hypergraph_from_list(sequence, n):
  """Returns the hypergraph from a list of coefficients."""
  hypergraph = {}
  idx_count = 0
  for a in range(n):
    for b in range(n):
      if a < b:
        if sequence[idx_count] > 0.5:
          hypergraph[(a, a, b)] = 1
        idx_count += 1
      elif b < a:
        if sequence[idx_count] > 0.5:
          hypergraph[(b, a, a)] = 1
        idx_count += 1
  for a in range(n):
    for b in range(a + 1, n):
      for c in range(b + 1, n):
        if sequence[idx_count] > 0.5:
          hypergraph[(a, b, c)] = 1
        idx_count += 1
  return hypergraph


def remove_k4s(hypergraph, n):
  """Removes k4s from the hypergraph."""
  new_hypergraph = hypergraph.copy()

  # a a b b
  for a in range(n):
    for b in range(a + 1, n):
      if (a, a, b) in new_hypergraph and (a, b, b) in new_hypergraph:
        del new_hypergraph[(a, a, b)]

  # a a b c
  for a in range(n):
    for b in range(a + 1, n):
      for c in range(b + 1, n):
        if (
            (a, a, b) in new_hypergraph
            and (a, a, c) in new_hypergraph
            and (a, b, c) in new_hypergraph
        ):
          del new_hypergraph[(a, a, c)]

  # b a a c
  for a in range(n):
    for b in range(a):
      for c in range(a + 1, n):
        if (
            (b, a, a) in new_hypergraph
            and (a, a, c) in new_hypergraph
            and (b, a, c) in new_hypergraph
        ):
          del new_hypergraph[(b, a, c)]

  # b c a a
  for a in range(n):
    for b in range(a):
      for c in range(b + 1, a):
        if (
            (c, a, a) in new_hypergraph
            and (b, a, a) in new_hypergraph
            and (b, c, a) in new_hypergraph
        ):
          del new_hypergraph[(b, a, a)]

  # a b c d
  for a in range(n):
    for b in range(a + 1, n):
      for c in range(b + 1, n):
        for d in range(c + 1, n):
          if (
              (a, b, c) in new_hypergraph
              and (a, b, d) in new_hypergraph
              and (a, c, d) in new_hypergraph
              and (b, c, d) in new_hypergraph
          ):
            del new_hypergraph[(a, c, d)]

  return new_hypergraph


def get_score(hypergraph, weights):
  """Returns the score for a hypergraph."""
  score = 0
  for key in hypergraph:
    if key[0] == key[1] or key[1] == key[2] or key[2] == key[0]:
      coeff = 3.0
    else:
      coeff = 6.0
    score += coeff * weights[key[0]] * weights[key[1]] * weights[key[2]]
  return score


def evaluate_sequences(sequence, weights) -> float:
  """Evaluates a sequence of coefficients."""
  n = len(weights)
  weights = np.array(weights)
  weights = np.clip(weights, 0, 1)
  # normalize so that sum(weights) = 1
  weights /= np.sum(weights)
  hypergraph = get_hypergraph_from_list(sequence, n)
  hypergraph = remove_k4s(hypergraph, n)
  score = get_score(hypergraph, weights)
  return score


def format_feedback_repr(feedback):
  """Formats feedback dictionary for representation in code."""
  formatted_feedback = {}
  np.set_printoptions(threshold=np.inf)
  for key, value in feedback.items():
    if isinstance(value, np.ndarray):
      repr_str = repr(value)  # Get repr string (e.g., "array([[...], [...]])")
      cleaned_repr_str = re.sub(r'[\n\s]+', ' ', repr_str)  # Clean up

      # Remove the leading "array(" and trailing ")" from repr string, then wrap
      # with "np.array(...)"
      array_content = cleaned_repr_str[
          6:-1
      ]  # Extract content inside "array(...)"

      if np.iscomplexobj(value):
        formatted_feedback[key] = (  # Use extracted content in np.array
            f'np.array({array_content}, dtype=np.complex128)'
        )
      elif not np.issubdtype(value.dtype, np.inexact):
        formatted_feedback[key] = f'np.array({array_content}, dtype=np.float64)'
      else:
        formatted_feedback[key] = f'np.array({array_content})'

    elif isinstance(value, list):
      formatted_feedback[key] = repr(value)  # Use standard repr for lists
    else:
      formatted_feedback[key] = repr(value)
  return formatted_feedback


def evaluate(weights_length) -> tuple[dict[str, float], dict[str, str]]:
  """Returns the numerical bound for the polygons if valid, or 0 if invalid."""
  result = {}
  feedback = {}
  n = weights_length
  sequence_length = (-1 + n) * n * (4 + n) // 6
  best_sequence, best_weights = search_for_best_sequence(
      sequence_length, weights_length
  )
  result['score'] = evaluate_sequences(best_sequence, best_weights)

  best_weights = np.array(best_weights)
  best_weights = np.clip(best_weights, 0, 1)
  best_weights /= np.sum(best_weights)

  feedback['best_sequence'] = best_sequence
  feedback['best_weights'] = best_weights
  feedback['best_score_found'] = result['score']
  feedback = format_feedback_repr(feedback)
  return result, feedback

In [ ]:
#@title Initial program

import itertools
import logging
import time
from scipy import integrate
import numpy as np
from scipy import optimize
import re
import numba

import warnings
from typing import Any, Callable, Mapping

njit = numba.njit



def search_for_best_sequence(sequence_length, weights_length):
  """Evolutionary search for finding the best coefficient sequence.

  Args:
    sequence_length: The length of the coefficient sequence.
    weights_length: The length of the weights vector.

  Returns:
    The best coefficient sequence found.
  """

  variable_name = f'best_sequence_{weights_length}'
  weights_variable_name = f'best_weights_{weights_length}'
  if variable_name in globals():
    best_sequence = globals()[variable_name]
    best_weights = globals()[weights_variable_name]
  else:
    best_sequence = np.random.randint(0, 1, sequence_length)
    best_weights = np.random.rand(weights_length)
  curr_sequence = best_sequence.copy()
  curr_weights = best_weights.copy()
  best_score = evaluate_sequences(best_sequence, best_weights)
  start_time = time.time()
  eval_count = 0
  total_time = 999
  while time.time() - start_time < total_time:
    eval_count += 1
    random_idx = np.random.randint(0, sequence_length)
    curr_sequence[random_idx] = 1 - curr_sequence[random_idx]
    random_idx = np.random.randint(0, weights_length)
    curr_weights[random_idx] = np.random.rand()
    score = evaluate_sequences(curr_sequence, curr_weights)
    if score > best_score:
      best_score = score
      best_sequence = curr_sequence.copy()
      best_weights = curr_weights.copy()
      print(f'New best score: {best_score:.6f}')
      print(f'Iteration {eval_count}, Best Score: {best_score:.6f}')

    if np.random.rand() < 0.2:
      curr_sequence = best_sequence.copy()

  return best_sequence, best_weights

**Prompt used**

Act as an expert software developer and optimization specialist specializing in creating arrays  with certain properties.
Your task is to generate two lists: the first one consisting of zeros and ones, the second one consisting of weights between 0 and 1. Your two lists will be evaluated by a mystery evaluation function that you are allowed to run yourself, but unfortunately we don't know what exactly it does.

Your task is to write a search function that searches for the two best lists. Your function will have 1000 seconds to run, and after that it has to have returned the best lists it found. If after 1000 seconds it has not returned anything, it will be terminated with negative infinity points. All numbers in your sequences have to be positive or zero.

You may code up any search method you want, and you are allowed to call the evaluate_sequences() function as many times as you want. You have access to it, you don't need to code up the evaluate_sequences() function. The score to beat is 0.5555. Good luck!



## What AlphaEvolve found

AlphaEvolve quickly discovered the known $5/9$ construction (using three vertices of equal weight $1/3$ with a specific set of edges) typically within one evolution step, matching the conjectured lower bound $C \geq 5/9$. Beyond that, it did not find any better constructions.